# EgoPHI -- Inspect predictions on a single frame

Pick a sequence/frame for ARCTIC and H2O, run the model once on each, and view the predictions in the script's own style and in the paper's figure style.


In [ ]:
import os
import sys

sys.path.insert(0, "/local/home/anilic/EgoPHI")

import numpy as np
import torch
import matplotlib.pyplot as plt
import pyvista as pv
from IPython.display import Image, display
from manopth.manolayer import ManoLayer

import config
import visualize_arctic
import visualize_h2o
from utils import render_prediction_figure


## 1. Configuration

Edit the sequence/frame below, then run the rest of the notebook.

In [ ]:
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

ARCTIC_PARTICIPANT = "s05"
ARCTIC_OBJECT = "box_grab_01"
ARCTIC_FRAME = None

H2O_PARTICIPANT = "subject4_ego"
H2O_OBJECT = "h1/0/cam4"
H2O_FRAME = None


## 2. Run inference

Loads both models and runs one forward pass per dataset on the chosen frame.

In [ ]:
device = torch.device(DEVICE)

arctic_model = visualize_arctic.load_model(device)
arctic_seq = visualize_arctic.prepare_sequence(ARCTIC_PARTICIPANT, ARCTIC_OBJECT, start_frame=ARCTIC_FRAME)
arctic_frame_position = ARCTIC_FRAME if ARCTIC_FRAME is not None else arctic_seq["start_frame"]
arctic_result = visualize_arctic.run_frame(arctic_seq, arctic_model, device, arctic_frame_position)
print(f"ARCTIC: {arctic_result['participant']}/{arctic_result['object_name']}, frame {arctic_result['frame_id']}")


In [ ]:
h2o_model = visualize_h2o.load_model(device)
h2o_seq = visualize_h2o.prepare_sequence(H2O_PARTICIPANT, H2O_OBJECT, start_frame=H2O_FRAME)
h2o_frame_position = H2O_FRAME if H2O_FRAME is not None else h2o_seq["start_frame"]
h2o_result = visualize_h2o.run_frame(h2o_seq, h2o_model, device, h2o_frame_position)
print(f"H2O: {h2o_result['participant']}/{h2o_result['object_key']}, frame {h2o_result['frame_id']}")


## 3. Script style

Same rendering `visualize_arctic.py`/`visualize_h2o.py` save to disk.

In [ ]:
import tempfile


def show_script_style(result, title):
    with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as f:
        render_prediction_figure(
            f.name, result["rgb_image"],
            result["hand_vertices"], result["hand_faces"], result["hand_contact"],
            result["hand_force_mag"], result["hand_force_dir"],
            result["object_vertices"], result["object_faces"], result["object_contact"],
            result["object_force_mag"], result["object_force_dir"],
            title=title,
        )
        display(Image(filename=f.name))


show_script_style(
    arctic_result,
    f"ARCTIC {arctic_result['participant']}/{arctic_result['object_name']} / {arctic_result['frame_id']}",
)


In [ ]:
show_script_style(
    h2o_result,
    f"H2O {h2o_result['participant']}/{h2o_result['object_key']} / {h2o_result['frame_id']}",
)


## 4. Paper style

Best-effort reconstruction of the paper's Figures 4/7/8 style (jet colormap, hands shown separately, no arrows).

In [ ]:
_mano_left_template = ManoLayer(
    side="left", use_pca=False, flat_hand_mean=True, ncomps=45, mano_root=config.MANO_ROOT
).th_v_template.squeeze(0).numpy()
_mano_right_template = ManoLayer(
    side="right", use_pca=False, flat_hand_mean=True, ncomps=45, mano_root=config.MANO_ROOT
).th_v_template.squeeze(0).numpy()


def _pv_faces(faces):
    faces = np.asarray(faces)
    return np.hstack([np.full((len(faces), 1), 3), faces]).astype(np.int64).flatten()


def render_paper_style(result, title=""):
    mesh_left = pv.PolyData(_mano_left_template, _pv_faces(result["faces_left"]))
    mesh_right = pv.PolyData(_mano_right_template, _pv_faces(result["faces_right"]))
    mesh_obj = pv.PolyData(result["object_vertices"], _pv_faces(result["object_faces"]))

    panels = [
        (mesh_left, result["left_contact"], "Left hand\ncontact"),
        (mesh_right, result["right_contact"], "Right hand\ncontact"),
        (mesh_obj, result["object_contact"], "Object\ncontact"),
        (mesh_left, result["left_force_mag"], "Left hand\nforce"),
        (mesh_right, result["right_force_mag"], "Right hand\nforce"),
        (mesh_obj, result["object_force_mag"], "Object\nforce"),
    ]

    plotter = pv.Plotter(shape=(2, 3), off_screen=True, window_size=(1500, 1000))
    for i, (mesh, values, panel_title) in enumerate(panels):
        row, col = divmod(i, 3)
        plotter.subplot(row, col)
        mesh = mesh.copy()
        mesh["value"] = np.clip(values, 0.0, 1.0)
        plotter.add_mesh(mesh, scalars="value", cmap="jet", clim=(0.0, 1.0),
                          show_scalar_bar=(col == 2), smooth_shading=True)
        plotter.add_text(panel_title, font_size=10)
        plotter.view_isometric()
    img = plotter.screenshot(return_img=True)
    plotter.close()

    plt.figure(figsize=(15, 10))
    plt.imshow(img)
    plt.axis("off")
    plt.title(title)
    plt.show()


render_paper_style(
    arctic_result,
    f"ARCTIC {arctic_result['participant']}/{arctic_result['object_name']} / {arctic_result['frame_id']} -- paper style",
)


In [ ]:
render_paper_style(
    h2o_result,
    f"H2O {h2o_result['participant']}/{h2o_result['object_key']} / {h2o_result['frame_id']} -- paper style",
)
